# Programming Agent Workflow

In [1]:
import os
import requests
from google.adk.agents import Agent, SequentialAgent
from vertexai.preview.reasoning_engines import AdkApp

# ==========================================
# 1. TOOL DEFINITION
# ==========================================

def search_web_info(query: str) -> str:
    """Performs a web search for real-time news, emergency shelters, and public updates.

    Args:
        query (str): Search query string.

    Returns:
        str: Summarized search output.
    """
    try:
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
        }
        url = f"https://html.duckduckgo.com/html/?q={requests.utils.quote(query)}"
        res = requests.get(url, headers=headers, timeout=10)
        if res.status_code == 200:
            from bs4 import BeautifulSoup
            soup = BeautifulSoup(res.text, "html.parser")
            results = soup.find_all("a", class_="result__snippet")
            output = [r.get_text(strip=True) for r in results[:3]]
            if output:
                return "Search Results:\n- " + "\n- ".join(output)
    except Exception as e:
        print(f"[Search Error]: {e}")

    return f"Search Information for '{query}': High-priority safety items include water (1 gal/person/day), non-perishable food, flashlights, battery-powered radio, first aid kit, and emergency contacts."


# ==========================================
# 2. WORKFLOW TEAM AGENTS (a, b, c)
# ==========================================

# Step A: Search Agent (Retrieves raw data to answer the query)
search_agent_step = Agent(
    name="SearchAgent",
    model="gemini-2.5-flash",
    description="Finds data and retrieves raw facts to answer the user's question.",
    instruction="""You are a research agent.
    Use the `search_web_info` tool to find factual, up-to-date data for the user's question.
    Provide a comprehensive initial answer based on your search results.""",
    tools=[search_web_info],
    output_key="initial_response"  # Stores output in state for CritiqueAgent
)

# Step B: Critique Agent (Evaluates initial response and suggests improvements)
critique_agent_step = Agent(
    name="CritiqueAgent",
    model="gemini-2.5-flash",
    description="Evaluates the initial response and suggests concrete improvements.",
    instruction="""You are an expert editor and safety reviewer.
    Review the initial answer provided by SearchAgent:
    "{initial_response}"

    Identify:
    1. Missing critical information or unaddressed aspects of the user's prompt.
    2. Any formatting, clarity, or organization issues.
    3. Additional safety steps or actionable takeaways that should be added.

    Provide a concise list of specific improvements. Do NOT write the final answer yourself.""",
    output_key="critique_feedback"  # Stores suggestions in state for RefineAgent
)

# Step C: Refine Agent (Rewrites response applying critique suggestions)
refine_agent_step = Agent(
    name="RefineAgent",
    model="gemini-2.5-flash",
    description="Rewrites and refines the response based on critique feedback.",
    instruction="""You are a technical editor.
    Review the initial draft:
    "{initial_response}"

    Review the suggested improvements:
    "{critique_feedback}"

    Rewrite the final response applying ALL suggestions. Ensure the output is well-structured, clear, highly readable, and uses bullet points and bold headers where appropriate."""
)


# ==========================================
# 3. WORKFLOW AGENT PIPELINE
# ==========================================

# SequentialAgent executes in strict order: Search -> Critique -> Refine
answer_team_workflow = SequentialAgent(
    name="AnswerRefinementTeam",
    description="Sequentially retrieves data, critiques response quality, and refines final output.",
    sub_agents=[search_agent_step, critique_agent_step, refine_agent_step]
)

# Root Greeter Agent (Primary entrypoint)
greeter_root_agent = Agent(
    name="GreeterAgent",
    model="gemini-2.5-flash",
    description="Greeter and primary entrypoint agent.",
    instruction="""You are Greeter, the primary entry point for user requests.
    Greet the user politely and pass their question to the `AnswerRefinementTeam` workflow to find, critique, and refine the answer.
    Return the final refined answer clearly.""",
    sub_agents=[answer_team_workflow]
)


# ==========================================
# 4. TEST EXECUTION
# ==========================================

def test_workflow_pipeline():
    app = AdkApp(agent=greeter_root_agent)
    session = app.create_session(user_id="workflow_tester")
    session_id = session.get("id") or session.get("name")

    test_query = "What should I pack in an emergency hurricane supply kit?"
    print("================ TESTING CHALLENGE 4 WORKFLOW PIPELINE ================")
    print(f"User Prompt: {test_query}\n")

    try:
        for event in app.stream_query(
            user_id="workflow_tester",
            session_id=session_id,
            message=test_query
        ):
            if isinstance(event, dict) and "content" in event:
                parts = event["content"].get("parts", [])
                for part in parts:
                    if "text" in part:
                        print(part["text"], end="")
    except Exception as e:
        print(f"[Workflow Execution Error]: {e}")

    print("\n" + "=" * 60)

# Execute Challenge 4 Test
test_workflow_pipeline()

/tmp/ipykernel_31694/3328514561.py:93: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  answer_team_workflow = SequentialAgent(
/usr/local/lib/python3.12/dist-packages/vertexai/preview/reasoning_engines/templates/adk.py:966: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self._tmpl_attrs["credential_service"] = InMemoryCredentialService()
/usr/local/lib/python3.12/dist-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()


================ TESTING CHALLENGE 4 WORKFLOW PIPELINE ================
User Prompt: What should I pack in an emergency hurricane supply kit?



/usr/local/lib/python3.12/dist-packages/google/adk/tools/function_tool.py:95: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  build_function_declaration(


Based on the repeated search results, high-priority items you should pack in an emergency hurricane supply kit include:

*   **Water:** At least 1 gallon per person per day.
*   **Non-perishable food:** Enough to last for several days.
*   **Flashlights:** With extra batteries.
*   **Battery-powered radio:** For receiving emergency broadcasts.
*   **First aid kit:** Essential for treating injuries.
*   **Emergency contacts:** Important phone numbers and information.

For a comprehensive and detailed list, it is highly recommended to consult official sources such as the American Red Cross or your local government's emergency management agency, as they often provide complete checklists and additional preparedness tips specific to your region.Here's a concise list of improvements:

1.  **Missing Critical Information:**
    *   The current list is far too brief for a "comprehensive" hurricane kit. It lacks essential categories like hygiene/sanitation, personal documents, pet supplies, tool